# 감정 분류 모델 학습 (한국어+영어 혼합)
**베이스 모델**: `kresnik/wav2vec2-large-xlsr-korean`  
**데이터**: 영어(CREMA-D/TESS/Ravdess/Savee) + 한국어(AIHub 감정 자유대화)  

## 사전 준비
1. Google Drive에 AIHub 전처리 결과 업로드
   - 데스크탑에서 preprocess.py 실행 후 생성된 `aihub_*.wav` 폴더 전체를
   - `MyDrive/se_emotion/korean_data/` 에 업로드
2. 런타임 → 런타임 유형 변경 → **T4 GPU** 또는 **A100** 선택

In [ ]:
# 1. 패키지 설치
!pip install transformers datasets soundfile librosa scikit-learn accelerate kaggle noisereduce -q

In [ ]:
# 2. Google Drive 마운트 (체크포인트 저장용)
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/se_emotion/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/se_emotion/model_best', exist_ok=True)
print('Drive 마운트 완료')

In [ ]:
# 3. 코드 클론
if not os.path.exists('/content/repo'):
    !git clone -b main https://github.com/woohyun212/SE-final-project.git /content/repo
else:
    !git -C /content/repo pull origin main
    print('repo 이미 있음, pull 완료')

In [ ]:
# 4. 영어 데이터 다운로드 (Kaggle)
# kaggle.json을 Drive에 업로드해두거나, 아래에 직접 입력
# 방법1: Drive에 kaggle.json 있는 경우 주석 해제
# !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json

# 방법2: 직접 입력
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'  # 본인 Kaggle 유저명으로 변경
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY'            # 본인 Kaggle API 키로 변경

os.makedirs('/content/data/raw', exist_ok=True)

if not os.path.exists('/content/data/raw/emotion-dataset-audio.zip'):
    print('영어 데이터 다운로드 중...')
    !kaggle datasets download -d seungjunlim/emotion-dataset-audio -p /content/data/raw
else:
    print('✓ 영어 데이터 zip 이미 있음')

from pathlib import Path
if not Path('/content/data/raw/emotion-dataset').exists():
    print('압축 해제 중...')
    !unzip -q /content/data/raw/emotion-dataset-audio.zip -d /content/data/raw/emotion-dataset
else:
    print('✓ 압축 해제 이미 됨')

In [ ]:
# 5. 영어 데이터 전처리
import sys
sys.path.insert(0, '/content/repo/ml')
from data.preprocess import _process_crema, _process_tess, _process_ravdess, _process_savee

emotion_root = Path('/content/data/raw/emotion-dataset')
src = emotion_root / os.listdir(emotion_root)[0]
dst = Path('/content/data/raw')

already = sum(len(list((dst / label).glob('*.wav'))) for label in ['angry','happy','sad'] if (dst/label).exists())
if already == 0:
    total = 0
    for name, fn in [('CREMA-D', _process_crema), ('TESS', _process_tess), ('Ravdess', _process_ravdess), ('Savee', _process_savee)]:
        n = fn(src, dst)
        print(f'  {name}: {n}개')
        total += n
    print(f'✓ 영어 전처리 완료: {total}개')
else:
    print(f'✓ 영어 전처리 이미 됨 ({already}개+)')

In [ ]:
# 6. 한국어 데이터 복사 (Google Drive → /content/data/raw)
# 데스크탑에서 preprocess.py --aihub-json ... --aihub-wav ... --dst <폴더> 실행 후
# 생성된 <폴더>/<label>/aihub_*.wav 파일들을 Google Drive의
# MyDrive/se_emotion/korean_data/<label>/ 에 미리 업로드해주세요.

korean_drive = Path('/content/drive/MyDrive/se_emotion/korean_data')

if korean_drive.exists():
    import shutil
    total_ko = 0
    for label_dir in korean_drive.iterdir():
        if label_dir.is_dir():
            target = dst / label_dir.name
            target.mkdir(parents=True, exist_ok=True)
            for wav in label_dir.glob('aihub_*.wav'):
                shutil.copy2(wav, target / wav.name)
                total_ko += 1
    print(f'✓ 한국어 데이터 복사 완료: {total_ko}개')
else:
    print('⚠ 한국어 데이터 없음 — 영어 데이터만으로 학습 진행')

In [ ]:
# 7. 데이터 현황 확인
labels = ['angry','disgust','fear','happy','neutral','sad','surprise']
total = 0
for label in labels:
    count = len(list((dst / label).glob('*.wav'))) if (dst / label).exists() else 0
    total += count
    print(f'  {label:10s}: {count:5d}개')
print(f'\n합계: {total}개')

In [ ]:
# 8. 학습
# wav2vec2-large-xlsr-korean 베이스, batch_size=8 (T4 16GB VRAM 기준)
# 체크포인트는 Google Drive에 저장 → 세션 끊겨도 이어서 학습 가능

os.chdir('/content/repo/ml')

checkpoint_dir = '/content/drive/MyDrive/se_emotion/checkpoints'

# 이전 체크포인트 있으면 이어서 학습
resume_flag = ''
checkpoints = list(Path(checkpoint_dir).glob('checkpoint-*')) if Path(checkpoint_dir).exists() else []
if checkpoints:
    latest = sorted(checkpoints, key=lambda p: int(p.name.split('-')[-1]))[-1]
    resume_flag = f'--resume_from_checkpoint {latest}'
    print(f'체크포인트 이어서 학습: {latest.name}')
else:
    print('처음부터 학습 시작')

!python -m train.train \
    --data_dir /content/data/raw \
    --output_dir {checkpoint_dir} \
    --base-model kresnik/wav2vec2-large-xlsr-korean \
    --epochs 10 \
    --batch_size 8 \
    --fp16 \
    --oversample \
    $resume_flag

In [ ]:
# 9. 최종 모델 Drive에 저장
import shutil
best_src = Path(checkpoint_dir) / 'best'
best_dst = Path('/content/drive/MyDrive/se_emotion/model_best')

if best_src.exists():
    shutil.copytree(best_src, best_dst, dirs_exist_ok=True)
    print(f'✓ 모델 저장 완료: {best_dst}')
else:
    print('❌ best 폴더 없음 — 학습이 완료되지 않았습니다')

In [ ]:
# 10. 로컬 다운로드 (선택)
from google.colab import files
shutil.make_archive('/content/model_best', 'zip', str(best_dst))
files.download('/content/model_best.zip')